<a href="https://colab.research.google.com/github/spavithra978/ZENDS-AI-Customer-Support-Copilot/blob/main/ZENDS_AI_Customer_Support_Copilot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install faker tqdm scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 51.9 MB/s eta 0:00:00


In [ ]:
import torch
print(torch.cuda.is_available())

True


In [ ]:
import random
import pandas as pd
from faker import Faker
from sklearn.utils import shuffle

fake = Faker()

def random_amount():
    return f"${random.randint(20,500)}"

def random_days():
    return random.randint(1,30)

rows = []

# 🔹 Billing (overlaps with refund + complaint)
billing_templates = [
    "I cancelled my service but was still charged {amount}.",
    "Why was I billed {amount} even after reporting service issues?",
    "My invoice shows {amount} but the service was down for {days} days.",
    "I was charged {amount} although my connection was not working.",
    "The billing amount of {amount} seems wrong and I want clarification."
]

# 🔹 Refund (overlaps with billing + complaint)
refund_templates = [
    "I cancelled my subscription and want my {amount} refunded.",
    "Since the service was down for {days} days, can I get a refund of {amount}?",
    "I was charged {amount} unfairly and need a refund.",
    "Please process my refund as the network was unstable.",
    "I am not satisfied and expect my {amount} back."
]

# 🔹 Technical (overlaps with complaint + billing)
technical_templates = [
    "My internet has been down for {days} days and I was still charged.",
    "The network keeps dropping and I cannot access services.",
    "Even after paying {amount}, my broadband is not working.",
    "The service disconnects frequently which is frustrating.",
    "Why is my connection unstable despite active subscription?"
]

# 🔹 Complaint (overlaps everywhere)
complaint_templates = [
    "I am extremely disappointed with the service and billing.",
    "The support team has not responded for {days} days regarding my issue.",
    "This delay and poor network quality is unacceptable.",
    "I am not satisfied with the way my refund request was handled.",
    "The overall experience has been frustrating."
]

# 🔹 Product Inquiry (overlaps with billing + technical)
product_templates = [
    "What features are included in your premium broadband plan?",
    "Does your enterprise plan cover service downtime compensation?",
    "Is there a plan that offers refund guarantees?",
    "Can you explain the difference between prepaid and postpaid plans?",
    "Do your plans include technical support?"
]

# 🔹 Other (real out-of-box but still telecom-related)
other_templates = [
    "What are your office working hours?",
    "How can I update my contact details?",
    "Do you offer internship opportunities?",
    "Where is your headquarters located?",
    "Can I change my registered email address?"
]

def generate_rows(templates, intent, count):
    for _ in range(count):
        template = random.choice(templates)
        text = template.format(amount=random_amount(), days=random_days())
        rows.append([text, intent])

generate_rows(billing_templates, "Billing", 3600)
generate_rows(refund_templates, "Refund", 3600)
generate_rows(technical_templates, "Technical", 3600)
generate_rows(complaint_templates, "Complaint", 3600)
generate_rows(product_templates, "Product Inquiry", 3600)
generate_rows(other_templates, "Other", 2000)

df = pd.DataFrame(rows, columns=["text", "intent"])
df = shuffle(df).reset_index(drop=True)

print("Dataset size:", len(df))
df.head()

Dataset size: 20000


,text,intent
0,My internet has been down for 15 days and I wa...,Technical
1,I was charged $248 although my connection was ...,Billing
2,Why is my connection unstable despite active s...,Technical
3,Why was I billed $275 even after reporting ser...,Billing
4,The overall experience has been frustrating.,Complaint


In [ ]:
print("\nSentiment Distribution:\n", df["sentiment"].value_counts())


In [ ]:
import random

def balanced_sentiment(intent):

    if intent == "Complaint":
        return random.choices(
            ["Angry", "Neutral", "Happy"],
            weights=[0.5, 0.3, 0.2]
        )[0]

    elif intent == "Technical":
        return random.choices(
            ["Angry", "Neutral", "Happy"],
            weights=[0.4, 0.4, 0.2]
        )[0]

    elif intent == "Billing":
        return random.choices(
            ["Neutral", "Happy", "Angry"],
            weights=[0.4, 0.4, 0.2]
        )[0]

    elif intent == "Refund":
        return random.choices(
            ["Neutral", "Happy", "Angry"],
            weights=[0.4, 0.4, 0.2]
        )[0]

    else:  # Product Inquiry
        return random.choices(
            ["Happy", "Neutral", "Angry"],
            weights=[0.6, 0.3, 0.1]
        )[0]

df["sentiment"] = df["intent"].apply(balanced_sentiment)

In [ ]:
df["sentiment"].value_counts()

,count
sentiment,
Happy,7688
Neutral,7140
Angry,5172


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "ramsrigouthamg/t5_paraphraser"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print("Model loaded on", device)

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model loaded on cuda


In [ ]:
import numpy as np

# Define how many rows we want to paraphrase
sample_size = 2000

# Randomly select 2000 row indices from dataset
indices = np.random.choice(df.index, sample_size, replace=False)

print("Selected rows for paraphrasing:", len(indices))

Selected rows for paraphrasing: 2000


In [ ]:


def paraphrase_sentence(sentence):

    text = "paraphrase: " + sentence + " </s>"

    encoding = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=64
    ).to(device)

    outputs = model.generate(
        **encoding,
        max_length=64,
        num_beams=5,
        num_return_sequences=1
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
from tqdm import tqdm

# Create a copy of original dataset
df_paraphrased = df.copy()

# Loop through selected 2000 rows
for idx in tqdm(indices):

    # Get original text
    original_text = df.loc[idx, "text"]

    # Generate paraphrased text
    new_text = paraphrase_sentence(original_text)

    # Replace only if output is valid
    if len(new_text.split()) > 3:
        df_paraphrased.at[idx, "text"] = new_text

100%|██████████| 2000/2000 [11:09<00:00,  2.99it/s]


In [ ]:
# Compare original vs paraphrased for 5 samples
for idx in indices[:15]:
    print("ORIGINAL:", df.loc[idx, "text"])
    print("PARAPHRASED:", df_paraphrased.loc[idx, "text"])
    print("="*70)

ORIGINAL: I am extremely disappointed with the service and billing.
PARAPHRASED: I am extremely disappointed with the service and billing.
ORIGINAL: This delay and poor network quality is unacceptable.
PARAPHRASED: This delay and poor network quality is unacceptable.
ORIGINAL: This delay and poor network quality is unacceptable.
PARAPHRASED: This delay and poor network quality is unacceptable.
ORIGINAL: Does your enterprise plan cover service downtime compensation?
PARAPHRASED: Does your enterprise plan cover service downtime compensation?
ORIGINAL: The overall experience has been frustrating.
PARAPHRASED: The overall experience has been frustrating.
ORIGINAL: The network keeps dropping and I cannot access services.
PARAPHRASED: The network keeps dropping and I cannot access services.
ORIGINAL: I was charged $340 although my connection was not working.
PARAPHRASED: I was charged $340 although my connection was not working.
ORIGINAL: Since the service was down for 20 days, can I get a r

In [ ]:
# Save final dataset to CSV file
df_paraphrased.to_csv("zends_final_dataset.csv", index=False)

print("Final dataset size:", len(df_paraphrased))

Final dataset size: 20000


In [ ]:
df_paraphrased.to_csv("zends_final_datasets.csv", index=False)

In [ ]:
# Install required libraries (only once in fresh notebook)
!pip install transformers datasets scikit-learn

In [ ]:
# Import basic libraries
import pandas as pd
import numpy as np
import torch

# Sklearn utilities
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score

# HuggingFace transformers
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

In [ ]:
import pandas as pd

# Load dataset
df = pd.read_csv("/content/drive/MyDrive/zends_final_datasets.csv")

# Check data
df.head()

,text,intent,sentiment
0,My internet has been down for 15 days and I wa...,Technical,Angry
1,I was charged $248 although my connection was ...,Billing,Happy
2,Why is my connection unstable despite active s...,Technical,Angry
3,Why was I billed $275 even after reporting ser...,Billing,Happy
4,The overall experience has been frustrating.,Complaint,Angry


In [ ]:
# Convert intent names into numeric labels
label_encoder = LabelEncoder()

df["label"] = label_encoder.fit_transform(df["intent"])

# Print label mapping for reference
print("Intent Label Mapping:")
for intent, label in zip(label_encoder.classes_, range(len(label_encoder.classes_))):
    print(intent, "->", label)

Intent Label Mapping:
Billing -> 0
Complaint -> 1
Other -> 2
Product Inquiry -> 3
Refund -> 4
Technical -> 5


In [ ]:
from sklearn.preprocessing import LabelEncoder

# Create label encoder
label_encoder = LabelEncoder()

# Fit encoder on intent column
df["intent_encoded"] = label_encoder.fit_transform(df["intent"])

# Check mapping
print(dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))

{'Billing': np.int64(0), 'Complaint': np.int64(1), 'Other': np.int64(2), 'Product Inquiry': np.int64(3), 'Refund': np.int64(4), 'Technical': np.int64(5)}


In [ ]:
import pickle

# Save label encoder
with open("label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

print("Label encoder saved successfully")

Label encoder saved successfully


In [ ]:
from google.colab import files
files.download("label_encoder.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Convert intent names into numeric labels
label_encoder = LabelEncoder()

df["label"] = label_encoder.fit_transform(df["intent"])

# Print label mapping for reference
print("Intent Label Mapping:")
for intent, label in zip(label_encoder.classes_, range(len(label_encoder.classes_))):
    print(intent, "->", label)

Intent Label Mapping:
Billing -> 0
Complaint -> 1
Other -> 2
Product Inquiry -> 3
Refund -> 4
Technical -> 5


In [ ]:
# Split dataset into 80% training and 20% testing
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["text"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]  # Maintains class balance
)

print("Training samples:", len(train_texts))
print("Testing samples:", len(test_texts))

Training samples: 16000
Testing samples: 4000


In [ ]:
# Load pre-trained DistilBERT tokenizer
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
print("Testing samples:", len(test_texts))

Testing samples: 4000


In [ ]:
# Convert text into tokenized format for model input
train_encodings = tokenizer(
    list(train_texts),
    truncation=True,
    padding=True,
    max_length=128   # Increased length for better context understanding
)

test_encodings = tokenizer(
    list(test_texts),
    truncation=True,
    padding=True,
    max_length=128
)

In [ ]:
# Custom dataset class for PyTorch
class IntentDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        # Convert each item into torch tensor
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# Create training and testing datasets
train_dataset = IntentDataset(train_encodings, train_labels.tolist())
test_dataset = IntentDataset(test_encodings, test_labels.tolist())

In [ ]:
# Load DistilBERT model for classification
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=6   # 6 intents including "Other"
)

# Move model to GPU
model.to("cuda")

In [ ]:
# Configure training settings
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,                 # 1 epoch to avoid overfitting
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",              # v5 uses eval_strategy
    save_strategy="epoch",
    logging_dir="./logs"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
# Function to compute evaluation metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average="weighted")
    }

In [ ]:
# Function to compute evaluation metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average="weighted")
    }

In [ ]:
# Create HuggingFace Trainer object
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
# Start training process
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.000644,0.000342,1.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1000, training_loss=0.038165993094444274, metrics={'train_runtime': 66.577, 'train_samples_per_second': 240.323, 'train_steps_per_second': 15.02, 'total_flos': 91077833088000.0, 'train_loss': 0.038165993094444274, 'epoch': 1.0})

In [ ]:
# Save trained model and tokenizer
model.save_pretrained("zends_intent_model")
tokenizer.save_pretrained("zends_intent_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('zends_intent_model/tokenizer_config.json',
 'zends_intent_model/tokenizer.json')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
model.save_pretrained("/content/drive/MyDrive/zends_intent_model")
tokenizer.save_pretrained("/content/drive/MyDrive/zends_intent_model")

In [ ]:
import pickle

with open("/content/drive/MyDrive/label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

In [ ]:
# Policy-based response templates

policy_responses = {
    "Billing": "According to ZEND billing policy, charges are applied as per subscription terms. If incorrect billing occurred, adjustments will be processed within 5–7 business days.",

    "Refund": "Refunds are processed as per ZEND refund policy. Refunds are eligible within the defined service period. Please allow 7 business days for processing.",

    "Technical": "We apologize for the inconvenience. As per our service policy, technical issues are addressed within SLA timeframes. Please restart your device and contact support if the issue persists.",

    "Complaint": "We regret your experience. Your concern will be escalated to our support team for review in accordance with our customer service policy.",

    "Product Inquiry": "Our plans include various features depending on the subscription tier. Please refer to our official product documentation for detailed plan comparison.",

    "Other": "We are unable to process this request as it is not covered under ZEND service policies. Please contact customer support for further clarification."
}

In [ ]:
import torch
import numpy as np

def predict_and_respond(user_query):

    # Tokenize input
    inputs = tokenizer(
        user_query,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    ).to("cuda")

    # Get model output
    with torch.no_grad():
        outputs = model(**inputs)

    # Get predicted class
    predicted_class_id = torch.argmax(outputs.logits, dim=1).item()

    # Convert label id back to intent name
    predicted_intent = label_encoder.inverse_transform([predicted_class_id])[0]

    # Get response from policy dictionary
    response = policy_responses[predicted_intent]

    return predicted_intent, response

In [ ]:
intent, reply = predict_and_respond("I was charged extra after cancelling my service.")
print("Predicted Intent:", intent)
print("Response:", reply)

Predicted Intent: Billing
Response: According to ZEND billing policy, charges are applied as per subscription terms. If incorrect billing occurred, adjustments will be processed within 5–7 business days.


In [ ]:
intent, reply = predict_and_respond("Do you offer internship opportunities?")
print("Predicted Intent:", intent)
print("Response:", reply)

Predicted Intent: Other
Response: We are unable to process this request as it is not covered under ZEND service policies. Please contact customer support for further clarification.


In [ ]:
!pip install transformers datasets scikit-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
df.to_csv("zends_final_datasets.csv", index=False)
from google.colab import files
files.download("zends_final_datasets.csv")

In [ ]:
model.save_pretrained("/content/drive/MyDrive/zends_intent_model")
tokenizer.save_pretrained("/content/drive/MyDrive/zends_intent_model")

In [ ]:
import pandas as pd
import numpy as np
import torch

from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder

from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/zends_final_datasets.csv")

df.head()

,text,intent,sentiment
0,My internet has been down for 15 days and I wa...,Technical,Angry
1,I was charged $248 although my connection was ...,Billing,Happy
2,Why is my connection unstable despite active s...,Technical,Angry
3,Why was I billed $275 even after reporting ser...,Billing,Happy
4,The overall experience has been frustrating.,Complaint,Angry


In [ ]:
label_encoder_sent = LabelEncoder()
df["sentiment_label"] = label_encoder_sent.fit_transform(df["sentiment"])

print("Sentiment Mapping:")
for sentiment, label in zip(label_encoder_sent.classes_, range(len(label_encoder_sent.classes_))):
    print(sentiment, "->", label)

Sentiment Mapping:
Angry -> 0
Happy -> 1
Neutral -> 2


In [ ]:
model_name = "cardiffnlp/twitter-roberta-base-sentiment"

tokenizer_sent = AutoTokenizer.from_pretrained(model_name)
model_sent = AutoModelForSequenceClassification.from_pretrained(model_name)

model_sent.to("cuda")

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_name = "distilbert-base-uncased-finetuned-sst-2-english"

tokenizer_sent = AutoTokenizer.from_pretrained(model_name)
model_sent = AutoModelForSequenceClassification.from_pretrained(model_name)

model_sent.to("cuda")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [ ]:
def predict_sentiment_sst2(text):

    inputs = tokenizer_sent(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    ).to("cuda")

    with torch.no_grad():
        outputs = model_sent(**inputs)

    probs = torch.softmax(outputs.logits, dim=1)
    prediction = torch.argmax(probs, dim=1).item()
    confidence = torch.max(probs).item()

    # Map labels
    if prediction == 0:
        sentiment = "Angry"
    else:
        sentiment = "Happy"

    # Optional neutral detection
    if confidence < 0.6:
        sentiment = "Neutral"

    return sentiment, confidence

In [ ]:
print(predict_sentiment_sst2("I am extremely disappointed with the service."))
print(predict_sentiment_sst2("Thank you for resolving my issue quickly."))
print(predict_sentiment_sst2("What are your office working hours?"))

('Angry', 0.9997683167457581)
('Happy', 0.999367892742157)
('Angry', 0.9951931238174438)


In [ ]:
def predict_sentiment(text):
    inputs = tokenizer_sent(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    ).to("cuda")

    with torch.no_grad():
        outputs = model_sent(**inputs)

    prediction = torch.argmax(outputs.logits, dim=1).item()
    return prediction

In [ ]:
sample_df = df.sample(20000, random_state=42).reset_index(drop=True)

predictions = []
for text in sample_df["text"]:
    predictions.append(predict_sentiment(text))

In [ ]:
mapping = {
    0: "Angry",
    2: "Neutral",
    1: "Happy"
}

predicted_sentiments = [mapping[p] for p in predictions]

sample_df["predicted_sentiment"] = predicted_sentiments

In [ ]:
accuracy = accuracy_score(sample_df["sentiment"], sample_df["predicted_sentiment"])
f1 = f1_score(sample_df["sentiment"], sample_df["predicted_sentiment"], average="weighted")

print("Sentiment Accuracy:", accuracy)
print("Sentiment F1 Score:", f1)

Sentiment Accuracy: 0.26355
Sentiment F1 Score: 0.10994199279806895


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import pandas as pd
from tqdm import tqdm

# Load SST-2 sentiment model
model_name = "distilbert-base-uncased-finetuned-sst-2-english"

tokenizer_sent = AutoTokenizer.from_pretrained(model_name)
model_sent = AutoModelForSequenceClassification.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model_sent.to(device)
model_sent.eval()

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [ ]:
def predict_sentiments_batch(text_list, batch_size=32):

    all_predictions = []

    for i in tqdm(range(0, len(text_list), batch_size)):

        batch_texts = text_list[i:i+batch_size]

        inputs = tokenizer_sent(
            batch_texts,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=128
        ).to(device)

        with torch.no_grad():
            outputs = model_sent(**inputs)

        probs = torch.softmax(outputs.logits, dim=1)
        predictions = torch.argmax(probs, dim=1)
        confidences = torch.max(probs, dim=1).values

        for pred, conf in zip(predictions, confidences):

            if pred.item() == 0:
                sentiment = "Angry"
            else:
                sentiment = "Happy"

            # Neutral detection using confidence threshold
            if conf.item() < 0.6:
                sentiment = "Neutral"

            all_predictions.append(sentiment)

    return all_predictions

In [ ]:
# Make sure dataset is loaded
# df = pd.read_csv("zends_final_dataset.csv")

predicted_sentiments = predict_sentiments_batch(df["text"].tolist(), batch_size=32)

df["predicted_sentiment"] = predicted_sentiments

100%|██████████| 625/625 [00:18<00:00, 34.44it/s]


In [ ]:
print(df["predicted_sentiment"].value_counts())

predicted_sentiment
Angry    20000
Name: count, dtype: int64


In [ ]:
from sklearn.metrics import accuracy_score, f1_score

accuracy = accuracy_score(df["sentiment"], df["predicted_sentiment"])
f1 = f1_score(df["sentiment"], df["predicted_sentiment"], average="weighted")

print("Sentiment Accuracy:", accuracy)
print("Sentiment F1 Score:", f1)

Sentiment Accuracy: 0.26355
Sentiment F1 Score: 0.10994199279806895


In [ ]:
# Install required libraries for RAG
!pip install sentence-transformers faiss-cpu pypdf

In [ ]:
# Import required libraries
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer


In [ ]:
# Import required libraries
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer


In [ ]:
# Structured policy knowledge base extracted from ZENDS Communications PDF
policy_documents = [

    # Billing
    "Billing Policy: Customers are billed monthly in advance according to their active subscription plan. If a customer cancels a service but still observes charges, the issue may relate to billing cycle timing. Late payments beyond 7 days may result in temporary suspension of services.",

    # Refund
    "Refund Policy: Customers are eligible for a full refund within 7 days of activation if usage is less than 10%. If a customer cancels a service early, refund eligibility depends on usage and activation status. Cloud services are strictly non-refundable after activation.",

    # Contracts
    "Contracts Policy: Individual users may cancel anytime without long-term lock-in. Enterprise customers are required to maintain a minimum 12-month contractual commitment.",

    # SLA
    "Service Level Agreement Policy: Individual users receive 98.5% uptime guarantee. Business users receive 99.5% uptime. Enterprise users receive 99.9% uptime guarantee with priority resolution.",

    # Cloud Services
    "Cloud Services Policy: ZENDCloud VM Basic, Pro, and Enterprise plans are available. Cloud virtual machines and storage services are provisioned instantly and are non-refundable after activation.",

    # Data Privacy
    "Data Privacy Policy: ZENDS complies with GDPR regulations, maintains ISO 27001 certification, and encrypts all customer data both at rest and in transit.",

    # Fair Usage
    "Fair Usage Policy: Unlimited mobile and broadband plans are capped at 1TB per month. Excess usage may result in speed throttling.",

    # Discounts
    "Discount Policy: Enterprise customers purchasing in bulk may receive up to 30% discount. Annual payment plans qualify for a 15% discount."
]

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Convert policy sections into embeddings
policy_embeddings = embedding_model.encode(policy_documents)

# Convert to float32 for FAISS
policy_embeddings = np.array(policy_embeddings).astype("float32")

print("Embeddings created successfully")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embeddings created successfully


In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 31.1 MB/s eta 0:00:00


In [ ]:
import faiss

# Get embedding dimension
dimension = policy_embeddings.shape[1]

# Create FAISS index
index = faiss.IndexFlatL2(dimension)

# Add embeddings to index
index.add(policy_embeddings)

print("FAISS index created with", index.ntotal, "policy sections")

FAISS index created with 8 policy sections


In [ ]:
def retrieve_policy_context(query, top_k=1):
    """
    Converts user query into embedding,
    searches FAISS index,
    returns most relevant policy section
    along with similarity score.
    """

    # Convert query to embedding
    query_embedding = embedding_model.encode([query])
    query_embedding = np.array(query_embedding).astype("float32")

    # Search FAISS index
    distances, indices = index.search(query_embedding, top_k)

    retrieved_section = policy_documents[indices[0][0]]
    similarity_score = distances[0][0]

    return retrieved_section, similarity_score

In [ ]:
def rag_generate_response(user_query, threshold=1.5):
    """
    Generates response with out-of-policy detection.
    """

    retrieved_section, score = retrieve_policy_context(user_query, top_k=1)

    # If similarity distance is too high → not relevant
    if score > threshold:

        return """
🤖 ZENDS AI Copilot Response:

We are unable to locate relevant information in our official policy regarding your request.

This query does not appear to be covered under current ZENDS service policies.

Please contact our customer support team for further clarification.
"""

    # Otherwise return retrieved policy
    return f"""
🤖 ZENDS AI Copilot Response:

Based on our official policy:

{retrieved_section}

If you require additional assistance, please contact ZENDS customer support.
"""

In [ ]:
print(rag_generate_response("What is the refund policy for cloud services?"))


🤖 ZENDS AI Copilot Response:

Based on our official policy:

Refund Policy: Customers are eligible for a full refund within 7 days if usage is less than 10%. Cloud services are non-refundable after activation.

If you require additional assistance, please contact ZENDS customer support.



In [ ]:
print(rag_generate_response("Do you sell laptops or smartphones?"))


🤖 ZENDS AI Copilot Response:

Based on our official policy:

Cloud Services: ZENDCloud VM Basic, Pro, and Enterprise plans are available. Pricing varies by country and customer type.

If you require additional assistance, please contact ZENDS customer support.



In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import pickle

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load intent model from Drive (adjust path if needed)
intent_model_path = "/content/drive/MyDrive/zends_intent_model"

tokenizer = AutoTokenizer.from_pretrained(intent_model_path)
model = AutoModelForSequenceClassification.from_pretrained(intent_model_path)
model.to(device)
model.eval()

# Load label encoder
with open("/content/drive/MyDrive/label_encoder.pkl", "rb") as f:
    label_encoder = pickle.load(f)

print("Intent model loaded successfully")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Intent model loaded successfully


In [ ]:
# Load SST-2 sentiment model
sent_model_name = "distilbert-base-uncased-finetuned-sst-2-english"

tokenizer_sent = AutoTokenizer.from_pretrained(sent_model_name)
model_sent = AutoModelForSequenceClassification.from_pretrained(sent_model_name)

model_sent.to(device)
model_sent.eval()

print("Sentiment model loaded successfully")

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Sentiment model loaded successfully


In [ ]:
def zends_ai_copilot(user_query, threshold=1.5):
    """
    Full AI Copilot Pipeline:
    - Intent Classification
    - Sentiment Detection
    - RAG Retrieval
    - Out-of-policy Detection
    """

    # -----------------------------
    # STEP 1: INTENT PREDICTION
    # -----------------------------
    intent_inputs = tokenizer(
        user_query,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        intent_outputs = model(**intent_inputs)

    intent_pred_id = torch.argmax(intent_outputs.logits, dim=1).item()
    predicted_intent = label_encoder.inverse_transform([intent_pred_id])[0]

    # -----------------------------
    # STEP 2: SENTIMENT PREDICTION
    # -----------------------------
    sent_inputs = tokenizer_sent(
        user_query,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        sent_outputs = model_sent(**sent_inputs)

    sent_probs = torch.softmax(sent_outputs.logits, dim=1)
    sent_pred = torch.argmax(sent_probs, dim=1).item()
    sent_conf = torch.max(sent_probs).item()

    if sent_pred == 0:
        predicted_sentiment = "Angry"
    else:
        predicted_sentiment = "Happy"

    if sent_conf < 0.6:
        predicted_sentiment = "Neutral"

    # -----------------------------
    # STEP 3: RAG RETRIEVAL
    # -----------------------------
    retrieved_section, score = retrieve_policy_context(user_query, top_k=1)

    # -----------------------------
    # STEP 4: OUT-OF-POLICY CHECK
    # -----------------------------
    if score > threshold:

        final_response = """
🤖 ZENDS AI Copilot Response:

We are unable to locate relevant information in our official policy database regarding your request.

This matter does not appear to fall within the scope of ZENDS' current service policies.

Please contact customer support for further assistance.
"""

    else:

        final_response = f"""
🤖 ZENDS AI Copilot Response:

Intent Detected: {predicted_intent}
Customer Sentiment: {predicted_sentiment}

Relevant Policy Information:
{retrieved_section}

If you require additional assistance, please contact ZENDS customer support.
"""

    return final_response

In [ ]:
print(zends_ai_copilot("I was charged even after cancelling my service."))

In [ ]:
print(zends_ai_copilot("I was charged even after cancelling my service."))


🤖 ZENDS AI Copilot Response:

Intent Detected: Billing
Customer Sentiment: Angry

Relevant Policy Information:
Billing Policy: Monthly billing is charged in advance. Enterprise customers receive consolidated invoices. Late payment beyond 7 days may result in service suspension.

If you require additional assistance, please contact ZENDS customer support.



In [ ]:
print(zends_ai_copilot("What is the refund policy for cloud services?"))


🤖 ZENDS AI Copilot Response:

Intent Detected: Product Inquiry
Customer Sentiment: Angry

Relevant Policy Information:
Refund Policy: Customers are eligible for a full refund within 7 days if usage is less than 10%. Cloud services are non-refundable after activation.

If you require additional assistance, please contact ZENDS customer support.



In [ ]:
print(zends_ai_copilot("Do you sell laptops or smartphones?"))


🤖 ZENDS AI Copilot Response:

Intent Detected: Other
Customer Sentiment: Angry

Relevant Policy Information:
Cloud Services: ZENDCloud VM Basic, Pro, and Enterprise plans are available. Pricing varies by country and customer type.

If you require additional assistance, please contact ZENDS customer support.



In [ ]:
print(zends_ai_copilot("I was charged even after cancelling my service."))


🤖 ZENDS AI Copilot Response:

Intent Detected: Billing
Customer Sentiment: Angry

Relevant Policy Information:
Billing Policy: Monthly billing is charged in advance. Enterprise customers receive consolidated invoices. Late payment beyond 7 days may result in service suspension.

If you require additional assistance, please contact ZENDS customer support.



In [ ]:
print(zends_ai_copilot("I was charged even after cancelling my service."))


🤖 ZENDS AI Copilot Response:

Intent Detected: Billing
Customer Sentiment: Angry

Relevant Policy Information:
Billing Policy: Customers are billed monthly in advance according to their active subscription plan. If a customer cancels a service but still observes charges, the issue may relate to billing cycle timing. Late payments beyond 7 days may result in temporary suspension of services.

If you require additional assistance, please contact ZENDS customer support.



In [ ]:
print(zends_ai_copilot("Do you sell gaming laptops?"))


🤖 ZENDS AI Copilot Response:

We are unable to locate relevant information in our official policy database regarding your request.

This matter does not appear to fall within the scope of ZENDS' current service policies.

Please contact customer support for further assistance.

